In [ ]:
SAE 204 : BDD - Groupe 

Dictionnaire des tables SQL

| table\_name | column\_name | data\_type | is\_nullable |
| :--- | :--- | :--- | :--- |
| controller | controllerid | integer | NO |
| controller | modelid | integer | NO |
| controller | serialnumber | text | NO |
| controlmeasurement | sensorid | integer | NO |
| controlmeasurement | controllerid | integer | NO |
| controlmeasurement | sensortimestamp | timestamp without time zone | NO |
| controlmeasurement | controltimestamp | timestamp without time zone | NO |
| controlmeasurement | controlvalue | double precision | NO |
| model | unit | text | NO |
| model | modelid | integer | NO |
| model | brand | text | NO |
| model | model | text | NO |
| model | name | text | NO |
| sensor | validity | interval | NO |
| sensor | sensorid | integer | NO |
| sensor | serialnumber | text | NO |
| sensor | position | text | NO |
| sensor | modelid | integer | NO |
| sensormeasurement | timestamp | timestamp without time zone | NO |
| sensormeasurement | sensorid | integer | NO |
| sensormeasurement | sensorvalue | double precision | NO |


In [ ]:
Tester la connexion avec la base de données

In [8]:
import psycopg

conn_params = {
    "host": "localhost",
    "dbname": "postgres",
    "user": "postgres",
    "password": "admin"
}

try:
    with psycopg.connect(**conn_params) as conn:
        print("Connection successful!")
        with conn.cursor() as cur:
            cur.execute("SELECT version();")
            print(cur.fetchone())
except psycopg.OperationalError as e:
    print(f"Connection failed: {e}")


Connection successful!
('PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35225, 64-bit',)


Script 1 — Erreurs entre mesures manuelles et automatiques :

In [1]:
import psycopg

conn_params = {
    "host": "localhost",
    "dbname": "postgres",
    "user": "postgres",
    "password": "admin"
}

try:
    with psycopg.connect(**conn_params) as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT sensormeasurement.sensorid,
                       controlmeasurement.controlvalue - sensormeasurement.sensorvalue AS diff
                FROM controlmeasurement
                JOIN sensormeasurement ON controlmeasurement.sensortimestamp = sensormeasurement.timestamp
                WHERE controlmeasurement.sensorid = 1
                ORDER BY controltimestamp
                LIMIT 50;
            """)
            for ligne in cur.fetchall():
                print(f"Capteur {ligne[0]} | Différence : {ligne[1]}")

except psycopg.OperationalError as e:
    print(f"Connexion échouée : {e}")

Capteur 1 | Différence : -0.6928256070072949
Capteur 1 | Différence : -0.42472458437324756
Capteur 1 | Différence : -0.19717837492203405
Capteur 1 | Différence : -0.05734016940635911
Capteur 1 | Différence : 0.14345425626137076
Capteur 1 | Différence : -0.37765161701976624
Capteur 1 | Différence : -0.38994523019952476
Capteur 1 | Différence : 0.4336675151979643
Capteur 1 | Différence : 0.7761312024213797
Capteur 1 | Différence : -0.15902853772262193
Capteur 1 | Différence : -0.0590040671430998
Capteur 1 | Différence : 0.4418077118033801
Capteur 1 | Différence : 0.5576625971798089
Capteur 1 | Différence : -0.02724000068365151
Capteur 1 | Différence : -0.0610747926891686
Capteur 1 | Différence : -0.14927230955878212
Capteur 1 | Différence : 0.3453796731956025
Capteur 1 | Différence : 0.565314546846384
Capteur 1 | Différence : 0.19167603339857386
Capteur 1 | Différence : 0.8958946595656778
Capteur 1 | Différence : -0.1490143137199928
Capteur 1 | Différence : -0.08535771270845571
Capteur 1

Script 2 — Moyenne et écart-type :

In [2]:
import psycopg

conn_params = {
    "host": "localhost",
    "dbname": "postgres",
    "user": "postgres",
    "password": "admin"
}

try:
    with psycopg.connect(**conn_params) as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT sensorid,
                       AVG(sensorvalue) AS Moyenne,
                       STDDEV(sensorvalue) AS Ecart_type
                FROM sensormeasurement
                WHERE sensorid = 1
                GROUP BY sensorid;
            """)
            ligne = cur.fetchone()
            print(f"Capteur {ligne[0]} | Moyenne : {ligne[1]:.2f} | Écart-type : {ligne[2]:.2f}")

except psycopg.OperationalError as e:
    print(f"Connexion échouée : {e}")

Capteur 1 | Moyenne : 0.00 | Écart-type : 1.24
